# GSM8K GRPO training on Colab

This notebook runs the `RL_GRPO_train` pipeline from Google Drive. It defaults to a 1-step smoke run; set `RUN_SMOKE = False` to launch the YAML configuration as-is.

In [ ]:
from pathlib import Path
import os

candidates = [
    Path('/content/drive/MyDrive/HPML_project'),
    Path('/content/drive/MyDrive/HPML_project/project'),
    Path.cwd(),
]
PROJECT_DIR = next((p for p in candidates if (p / 'RL_GRPO_train').exists() and (p / 'RL_common').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Cannot find project root. Mount Drive or edit PROJECT_DIR manually.')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
!pwd

## Tokens and SwanLab

For VSCode Colab extension, Colab Secrets may be unavailable. The default path below uses `getpass` and environment variables. If you run in the Colab web UI and want to use Secrets, set `USE_COLAB_SECRETS = True`.

In [ ]:
import getpass
import os

USE_COLAB_SECRETS = True

if USE_COLAB_SECRETS:
    from google.colab import userdata
    for key in ['HF_TOKEN', 'SWANLAB_API_KEY']:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
else:
    for key in ['HF_TOKEN', 'SWANLAB_API_KEY']:
        if not os.environ.get(key):
            value = getpass.getpass(f'{key} (press Enter to skip): ')
            if value:
                os.environ[key] = value

os.environ.setdefault('SWANLAB_PROJECT', 'gsm8k-grpo')
# Optional: set this if your SwanLab account uses a specific workspace.
# os.environ.setdefault('SWANLAB_WORKSPACE', 'your-workspace')

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

print('SWANLAB_PROJECT =', os.environ.get('SWANLAB_PROJECT'))
print('HF_TOKEN set =', bool(os.environ.get('HF_TOKEN')))
print('SWANLAB_API_KEY set =', bool(os.environ.get('SWANLAB_API_KEY')))

## Install dependencies

In [ ]:
!pip uninstall -y torchao
!pip install -q -U "trl>=0.29.0" "accelerate>=1.4.0" swanlab bitsandbytes sentencepiece
!pip install -q -e RL_common

import torch
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))

## Select config

`RUN_SMOKE=True` creates a tiny temporary config for checking the trainer, rewards, SwanLab logging, and best-adapter saving. For the real run, set it to `False` and edit the YAML directly.

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = PROJECT_DIR / 'RL_GRPO_train/configs/qwen25_3b_sft_grpo.yaml'
RUN_SMOKE = True

if RUN_SMOKE:
    cfg = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
    cfg['run']['name'] = cfg['run']['name'] + '_smoke'
    cfg['train_dataset']['limit'] = 8
    cfg['eval_dataset']['limit'] = 8
    cfg['eval']['every_steps'] = 1
    cfg['eval']['batch_size'] = 4
    cfg['eval']['max_new_tokens'] = 128
    cfg['reward']['max_completion_tokens'] = 128
    cfg['grpo']['max_steps'] = 1
    cfg['grpo']['logging_steps'] = 1
    cfg['grpo']['num_generations'] = 2
    cfg['grpo']['generation_batch_size'] = 4
    cfg['grpo']['per_device_train_batch_size'] = 2
    cfg['grpo']['gradient_accumulation_steps'] = 1
    cfg['grpo']['max_completion_length'] = 128
    cfg['grpo']['run_name'] = cfg['grpo']['run_name'] + '_smoke'
    ACTIVE_CONFIG = Path('/content/grpo_smoke.yaml')
    ACTIVE_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
else:
    ACTIVE_CONFIG = BASE_CONFIG

print('ACTIVE_CONFIG =', ACTIVE_CONFIG)
print(ACTIVE_CONFIG.read_text(encoding='utf-8')[:2000])

## Train

Outputs are written under `RL_GRPO_train/outputs/{resolved_run_name}`. Training metrics go to SwanLab through `report_to: swanlab`; greedy validation metrics are logged as `sft_val/*` by the callback.

In [ ]:
!accelerate launch --num_processes 1 RL_GRPO_train/train_grpo.py --config "{ACTIVE_CONFIG}"

## Inspect outputs

In [ ]:
import json
from rl_common.config import format_run_name, load_config, make_run_dir

cfg = load_config(str(ACTIVE_CONFIG))
cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)
run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
print('run_dir =', run_dir)
!find "{run_dir}" -maxdepth 2 -type f | sort

summary_path = run_dir / 'train_summary.json'
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8'))

best_path = run_dir / 'best_eval_results.json'
if best_path.exists():
    data = json.loads(best_path.read_text(encoding='utf-8'))
    print(json.dumps(data['summary'], indent=2, ensure_ascii=False))

## Optional final eval on official GSM8K test

Do not run this during training. This cell rewrites a temporary eval config so `adapter_path` points to the trained `final_adapter`, not the original SFT adapter.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    cfg = load_config(str(ACTIVE_CONFIG))
    cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)
    run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
    final_adapter = run_dir / 'final_adapter'
    if not final_adapter.exists():
        raise FileNotFoundError(f'Missing final adapter: {final_adapter}')
    cfg['model']['adapter_path'] = str(final_adapter)
    cfg['model']['tokenizer_name_or_path'] = str(final_adapter)
    cfg['run']['name'] = cfg['run']['resolved_name'] + '_final_test'
    eval_cfg = Path('/content/grpo_final_eval.yaml')
    eval_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    !python RL_GRPO_train/train_grpo.py --config "{eval_cfg}" --eval-only --final-test